# E2: Multi-head self-attention and tokenization on CIFAR-10 (replication notebook)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/broistg/deep-learning-CO5085/blob/main/E2/notebooks/E2.ipynb)

Thin notebook: all code lives in `E2/e2lib/` and `common/`. Two modes (set `MODE` in the next cell):

| `MODE` | What it does | Runtime | Time |
|---|---|---|---|
| `"load"` (default) | Takes the **committed results** (30 runs), regenerates the tables, curves and (optionally) the confusion matrix and error gallery, and **asserts that the tables equal the ones printed in the report** | free Colab, CPU is enough | a few minutes |
| `"rerun"` | Retrains **all 30 runs** from scratch (4 tokenizers x scratch/torch MSA x seeds 0,1,2 + positional-encoding ablation) into a fresh `E2/results/`; the published results are kept in `E2/results_published/` and compared with the rerun | GPU runtime | about 2.5 h on a laptop RTX 3050; expect a similar order on a Colab T4 (the notebook prints an estimate first) |

A rerun does not reproduce the numbers bit by bit (another GPU, cuDNN and mixed precision), so it is judged by the mean per configuration against the published standard deviation. The load mode is the exact check of the report. Protocol and decisions: `E2/README.md`.

In [ ]:
# 0. Settings
MODE = "load"            # "load" (minutes, CPU ok) or "rerun" (hours, GPU)
FETCH_WEIGHTS = True     # load mode: download the test logits from the GitHub Release (for the confusion matrix and error gallery); ~tens of MB
REPO_URL = "https://github.com/broistg/deep-learning-CO5085.git"
BRANCH = "main"

In [ ]:
# 1. Code: clone or update the repo, install the dependencies (torch is NOT reinstalled), put the repo on the path
import os, sys
ON_COLAB = "google.colab" in sys.modules
REPO_DIR = "/content/repo" if ON_COLAB else os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ON_COLAB and not os.path.isdir(REPO_DIR + "/.git"):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
if ON_COLAB:
    !git pull --ff-only
!pip install -q -r requirements/e.txt
!pip install -q -e .
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from common import notebook_utils as nu

In [ ]:
# 2. What this run will do
nu.banner(MODE, "E2")
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if MODE == "rerun":
    assert torch.cuda.is_available(), "rerun needs a GPU runtime (Runtime > Change runtime type)"
    h, n = nu.estimate_hours("E2/results")
    print(f"Estimate from the published runs: {n} runs, about {h:.1f} h of pure training time on the original laptop GPU (plus evaluation overhead).")

In [ ]:
# 3. Tests: the from-scratch MSA equals torch.nn.MultiheadAttention, tokenizer shapes, split hash
!python -m unittest discover -s E2/tests -v
!python -m unittest discover -s common/tests -v

## Load mode

In [ ]:
# 4a. (load) weights/logits from the Release, verified by SHA-256 (only needed for the confusion matrix and the error gallery)
if MODE == "load" and FETCH_WEIGHTS:
    try:
        nu.fetch_release("e2_artifacts.zip")
        !python -m common.cifar          # CIFAR-10 download + frozen split (hash must match common/cifar10_split.json)
    except Exception as e:
        print("WARNING: weights not available, the confusion matrix and error gallery will be skipped:", e)

In [ ]:
# 4b. (load) regenerate tables, curves, figures from E2/results/*.json into E2/regenerated (nothing committed is overwritten)
if MODE == "load":
    !python -m e2lib.analyze --out E2/regenerated

In [ ]:
# 4c. (load) check: regenerated tables == committed tables (byte for byte), summary.json equal
if MODE == "load":
    nu.compare_outputs("E2/regenerated", "E2/results")

## Rerun mode

In [ ]:
# 5a. (rerun) dry run on synthetic data, then move the published results aside
if MODE == "rerun":
    !python -m e2lib.train --tokenizer patch4 --attn scratch --seed 0 --fake --smoke
    nu.stash_published("E2")

In [ ]:
# 5b. (rerun) all 30 runs. Resumable: if Colab disconnects, re-run the notebook (MODE="rerun") and finished runs are skipped.
if MODE == "rerun":
    !python -m e2lib.plan

In [ ]:
# 5c. (rerun) compare with the published results, then regenerate the tables from the new runs
if MODE == "rerun":
    nu.compare_means("E2/results", "E2/results_published")
    nu.compare_runs("E2/results", "E2/results_published")
    !python -m e2lib.analyze --out E2/regenerated

## Figures and tables

In [ ]:
# 6. Show the regenerated figures and the tables
from pathlib import Path
nu.show_pngs("E2/regenerated/figures")
for f in sorted(Path("E2/regenerated").glob("tab_*.tex")):
    print("---", f.name); print(f.read_text(encoding="utf8"))

Published results are never overwritten by this notebook. To put them back after a rerun: `nu.restore_published("E2")`.
To put your own rerun into the report, copy `E2/regenerated/tab_*.tex` and the figures where the report expects them (Appendix "Tái lập kết quả E2").